# Ratio 5 Initializations -- Extra (HEA & QAOA) | DRIVER SONG SONG

Chay song song `ratio_worker.compute_one` tren nhieu tien trinh, **checkpoint theo tung
config** `(init, ham, ansatz, n)` vao `ckpt_extra/`, **resume** duoc neu ngat.
Cuoi cung gop -> `ratio_5inits_extra.json` (schema **y het** ban goc:
`init -> ham -> {'hea'|'qaoa'} -> {str(n): [50 ratios]}`).

> Khong dung lai/khong sua notebook goc dang chay. Bo nay doc lap.
> Logic ratio bam dung `ratio_ising/maxcut/partition`; chi khac: seed TAT DINH theo task
> (thay `np.random.seed(42)` toan cuc) -> tai lap duoc & doc lap giua worker.


In [1]:
import os, sys, json, time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np

import ratio_worker as W          # module worker (cung thu muc)
from ratio_worker import compute_one

BASE = W._HERE
CKPT = os.path.join(BASE, "ckpt_extra")
os.makedirs(CKPT, exist_ok=True)

# --- Luoi (giong ban goc) ---
inits          = ['normal', 'uniform', 'pi4', 'he', 'glorot']
hams           = ['ising', 'maxcut', 'partition']
ansatz_configs = [('hea', 2), ('qaoa', 2)]
qubits         = [3, 6, 9, 12, 15]
n_problem      = 50

MAX_WORKERS    = 8   # GIU tien trinh cu chay: 7 worker + 1 tien trinh cu = 8 nhan vat ly.
                     # Neu da dung tien trinh cu, doi thanh 8.

print("logical cores:", os.cpu_count(), "| workers:", MAX_WORKERS)
print("checkpoint dir:", CKPT)

logical cores: 16 | workers: 8
checkpoint dir: c:\Users\Dao Duy Tung\Documents\Python\newbie\IQP_Optimization\ckpt_extra


## Checkpoint helpers (atomic write, resume-safe)

In [2]:
def ckpt_path(init, ham, ansatz, n):
    return os.path.join(CKPT, f"{init}__{ham}__{ansatz}__n{n}.json")

def config_done(init, ham, ansatz, n):
    p = ckpt_path(init, ham, ansatz, n)
    if not os.path.exists(p):
        return False
    try:
        with open(p) as f:
            data = json.load(f)
        return isinstance(data, list) and len(data) == n_problem \
               and all(x is not None for x in data)
    except Exception:
        return False

def save_ckpt(init, ham, ansatz, n, ratios):
    p = ckpt_path(init, ham, ansatz, n)
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(ratios, f)
    os.replace(tmp, p)            # atomic: khong bao gio co file 1/2 chung


## Smoke test (1 task) -- kiem tra worker truoc khi mo pool
Chay nhanh 1 task nho de chac chan import/tinh toan OK (neu loi se hien ngay o day,
khong phai doi trong ProcessPool).

In [3]:
t = ('normal', 'ising', 'hea', 2, 3, 0)
print("smoke:", compute_one(t))


smoke: (('normal', 'ising', 'hea', 2, 3, 0), 0.5570539882593717)


## Build task list (BO QUA config da co checkpoint -> resume)

In [4]:
tasks = []
skipped = 0
total_cfg = len(inits) * len(hams) * len(ansatz_configs) * len(qubits)
for init in inits:
    for ham in hams:
        for ansatz, spec in ansatz_configs:
            for n in qubits:
                if config_done(init, ham, ansatz, n):
                    skipped += 1
                    continue
                for j in range(n_problem):
                    tasks.append((init, ham, ansatz, spec, n, j))

remaining_cfg = len({(t[0], t[1], t[2], t[4]) for t in tasks})
print(f"configs da xong (bo qua): {skipped}/{total_cfg}")
print(f"configs con lai        : {remaining_cfg}")
print(f"tasks con lai          : {len(tasks)}")


configs da xong (bo qua): 121/150
configs con lai        : 29
tasks con lai          : 1450


## Chay pool (dispatch dong; checkpoint ngay khi 1 config du 50 instance)
Ngat giua chung van an toan: config nao du 50 da co file, chay lai chi lam tiep config con thieu.

In [5]:
buf = defaultdict(dict)          # (init,ham,ansatz,n) -> {j: ratio}
done_cfg = 0
t0 = time.time()

if tasks:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(compute_one, t) for t in tasks]
        for k, fut in enumerate(as_completed(futs), 1):
            (init, ham, ansatz, spec, n, j), r = fut.result()
            d = buf[(init, ham, ansatz, n)]
            d[j] = r
            if len(d) == n_problem:                 # config day -> ghi checkpoint
                ratios = [d[i] for i in range(n_problem)]
                save_ckpt(init, ham, ansatz, n, ratios)
                del buf[(init, ham, ansatz, n)]
                done_cfg += 1
                el = time.time() - t0
                eta = el / done_cfg * (remaining_cfg - done_cfg)
                print(f"[{el/60:6.1f}m] cfg {done_cfg}/{remaining_cfg} "
                      f"{init:8s} {ham:9s} {ansatz:4s} n={n:<2d} "
                      f"mean={np.mean(ratios):.3f} | task {k}/{len(tasks)} "
                      f"| ETA ~{eta/60:.0f}m", flush=True)
    print(f"\nXONG tat ca task trong {(time.time()-t0)/3600:.2f} h")
else:
    print("Khong con task -> tat ca config da co checkpoint. Sang cell gop.")


[  36.8m] cfg 1/29 glorot   ising     hea  n=9  mean=0.665 | task 98/1450 | ETA ~1031m
[  38.5m] cfg 2/29 glorot   ising     hea  n=12 mean=0.772 | task 148/1450 | ETA ~520m
[  41.8m] cfg 3/29 he       partition qaoa n=15 mean=0.250 | task 174/1450 | ETA ~362m
[  44.0m] cfg 4/29 glorot   ising     hea  n=15 mean=0.853 | task 228/1450 | ETA ~275m
[  44.3m] cfg 5/29 glorot   ising     qaoa n=3  mean=0.398 | task 250/1450 | ETA ~213m
[  45.9m] cfg 6/29 glorot   ising     qaoa n=6  mean=0.495 | task 300/1450 | ETA ~176m
[  49.1m] cfg 7/29 glorot   ising     qaoa n=9  mean=0.538 | task 350/1450 | ETA ~154m
[  55.1m] cfg 8/29 glorot   ising     qaoa n=12 mean=0.539 | task 400/1450 | ETA ~145m
[  76.1m] cfg 9/29 glorot   maxcut    hea  n=3  mean=0.099 | task 496/1450 | ETA ~169m
[  76.7m] cfg 10/29 glorot   maxcut    hea  n=6  mean=0.120 | task 548/1450 | ETA ~146m
[  77.6m] cfg 11/29 glorot   maxcut    hea  n=9  mean=0.178 | task 598/1450 | ETA ~127m
[  78.8m] cfg 12/29 glorot   maxcut    he

## Gop checkpoint -> ratio_5inits_extra.json (schema y het ban goc)
Chi xuat khi DU tat ca config. Neu thieu, in ra danh sach de biet chay tiep cai nao.

In [7]:
merged = {}
missing = []
for init in inits:
    merged[init] = {}
    for ham in hams:
        merged[init][ham] = {}
        for ansatz, spec in ansatz_configs:
            merged[init][ham][ansatz] = {}
            for n in qubits:
                p = ckpt_path(init, ham, ansatz, n)
                if not os.path.exists(p):
                    missing.append((init, ham, ansatz, n))
                    continue
                with open(p) as f:
                    merged[init][ham][ansatz][str(n)] = json.load(f)

if missing:
    print(f"THIEU {len(missing)} config -> CHUA xuat file cuoi.")
    print("Vi du:", missing[:8])
else:
    out = os.path.join(BASE, "ratio_5inits_extra_parallel.json")
    with open(out, "w") as f:
        json.dump(merged, f, indent=4)
    print("Da xuat", out)
    print("Gio co the chay cell 'Merge + ve 5 ansatz' trong notebook Extra goc.")


Da xuat c:\Users\Dao Duy Tung\Documents\Python\newbie\IQP_Optimization\ratio_5inits_extra_parallel.json
Gio co the chay cell 'Merge + ve 5 ansatz' trong notebook Extra goc.
